# Introduction

In this notebook, I will be working on the deployment of a machine learning model in order to predict NCAA player future impact in the NBA. 
The goal here, is to:

* Automate the quick preprocessing steps (unstructured data, missing values) to have a workable dataframe

* Automate the modelling to get quick evaluation of all of the models by making varying: scaling, target, test size, model

*  When we get the best model : we make varying the parameters to avoid overfitting and get the 'perfect' model


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.preprocessing import OneHotEncoder

In [2]:
HOME = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire"
W_DB = r"\Database\Working db"
STACK = r"\Thesis - Code\Database Updates\Database stack"

# Preprocessing

In [3]:
working_df = pd.read_excel(HOME + STACK + r"\looping_df_1744186780.4209929.xlsx")

In [12]:
#Vectorizing dataframe based on target chosen 
features = [col for col in working_df.columns if 'features' in col]
metric = 'WS/48'
season = 1
target = f'{metric}-{season}_target'
working_df[[target] + features][working_df[target].notnull()]

,WS/48-1_target,Unnamed: 0.1_features,Unnamed: 0_features,Player_features,#_features,Class_features,Pos_x_features,Height_features,Weight_features,Hometown_features,...,FTA_opp_features,FT%_opp_features,ORB_opp_features,TRB_opp_features,AST_opp_features,STL_opp_features,BLK_opp_features,TOV_opp_features,PF_opp_features,draft_season_features
0,0.112,6752,2693,Aaron Brooks,0.0,SR,G,6-0,161,"Seattle, WA",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
1,0.060,33627,142,Aaron Gordon,11.0,FR,F,6-9,225,"San Jose, CA",...,698.0,0.716,369.0,1181.0,347.0,178.0,118.0,471.0,699.0,15
2,0.066,6836,2777,Aaron Gray,33.0,SR,C,7-0,270,"Emmaus, PA",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
3,-0.014,39716,1803,Aaron Harrison,2.0,SO,G,6-6,212,"Richmond, TX",...,672.0,0.659,463.0,1215.0,301.0,182.0,94.0,533.0,778.0,16
4,-0.306,66783,2329,Aaron Henry,0.0,JR,F,6-6,210,"Indianapolis, IN",...,599.0,0.716,265.0,959.0,371.0,181.0,86.0,295.0,498.0,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,0.104,60131,143,Zeke Nnaji,22.0,FR,F,6-11,240,"Lakeville, MN",...,632.0,0.733,296.0,1083.0,363.0,155.0,99.0,465.0,614.0,21
954,0.048,55067,3890,Zhaire Smith,2.0,FR,G,6-5,195,"Garland, TX",...,762.0,0.703,368.0,1199.0,377.0,222.0,138.0,554.0,766.0,19
955,0.080,68298,3844,Ziaire Williams,3.0,FR,F,6-8,185,"Lancaster, CA",...,435.0,0.749,280.0,905.0,342.0,189.0,97.0,388.0,494.0,22
956,0.141,56552,986,Zion Williamson,1.0,FR,F,6-7,285,"Spartanburg, SC",...,579.0,0.696,441.0,1348.0,479.0,243.0,121.0,540.0,659.0,20


for some players 'Advanced' is empty (assuming that it is player before 2010) advanced is ok, but the rest (totals PProd, ...) are errors

In [10]:
#Unstructured data
text_data = [
    "Class", #Dummy 
    "Pos_x", #Dummy
    "average_Awards", #Dummy & flag
    "Strength", #Scouting report
    "Weakness", #Scouting report
    "RSCI Top 100", #Rankings represent the player's final standing within his high school class in a given year/Dummy
]
#Biometric data that I will use
biometric_data = [
    "Height", #Won't change
    "Weight", #Won't change
    "Draft_pos", #Proxy ?
]
#Everything that I want to delete
adv_to_del = [
    "advanced_Awards", "totals_PER", "totals_TS%", "totals_3PAr",	"totals_FTr", "totals_PProd", "totals_ORB%", "totals_DRB%", "totals_TRB%", "totals_AST%", "totals_STL%", "totals_BLK%", "totals_TOV%", "totals_USG%", "totals_OWS", "totals_DWS", "totals_WS", "totals_WS/40", "totals_OBPM", "totals_DBPM", "totals_BPM",
	"G", "MP", "FG", "FGA", "FG%", "2P", "2PA",	"2P%", "3P", "3PA",	"3P%", "FT", "FTA",	"FT%", "ORB", "DRB", "TRB",	"AST", "STL", "BLK", "TOV",	"PF", "PTS"
]
team_to_del = [
    "G_opp", "W_opp", "L_opp", "W-L%_opp", "SRS_opp", "SOS_opp", "W.1_opp", "L.1_opp", 
    "W.2_opp", "L.2_opp", "W.3_opp", "L.3_opp", "Tm._opp", "Opp._opp", "MP_opp" #Just a replicate of the other part from team
]
id = [
    "UID_scout", #Won't use
    "team_for_merging", #Won't use
    "Name_for_merging", #Won't use
    "Name" #Won't use
]
useless = [
    "player_id",
    "Player",	
    "Unnamed: 0.1",	
    "Unnamed: 0",
    "Summary", 
    "average_Rk", 
    "average_Pos",
    "#", #Uniform number
    "totals_Rk", "per_40_Rk",
    "totals_Pos", "totals_G", "totals_GS",
    "per_40_Pos", "per_40_G", "per_40_GS",
    "totals_MP", "per_40_MP",
    "Unnamed: 0_team",
    "Unnamed: 24", 
    "Unnamed: 140",
    "advanced_Rk", "advanced_Pos", "advanced_G", "advanced_GS", "advanced_MP",
    "Pos_y",
    "totals_Awards",
    "per_40_Awards",
    "del1", "del2", "del3", "del4", "del5", 
    "del1_opp", "del2_opp", "del3_opp", "del4_opp", "del5_opp",
    "UID_scout", #Won't use
    "team_for_merging", #Won't use
    "Name_for_merging", #Won't use
    "Name", #Won't use
    "G_opp", "W_opp", "L_opp", "W-L%_opp", "SRS_opp", "SOS_opp", "W.1_opp", "L.1_opp", 
    "W.2_opp", "L.2_opp", "W.3_opp", "L.3_opp", "Tm._opp", "Opp._opp", "MP_opp",
    "advanced_Awards", "totals_PER", "totals_TS%", "totals_3PAr",	"totals_FTr", "totals_PProd", "totals_ORB%", "totals_DRB%", "totals_TRB%", "totals_AST%", "totals_STL%", "totals_BLK%", "totals_TOV%", "totals_USG%", "totals_OWS", "totals_DWS", "totals_WS", "totals_WS/40", "totals_OBPM", "totals_DBPM", "totals_BPM",
	"G", "MP", "FG", "FGA", "FG%", "2P", "2PA",	"2P%", "3P", "3PA",	"3P%", "FT", "FTA",	"FT%", "ORB", "DRB", "TRB",	"AST", "STL", "BLK", "TOV",	"PF", "PTS",
    "height", #Won't use
    "Ht", #From scouting
    "Wt", #From scouting
    "season", #Won't use
    "draft_season", #Won't use 
    "Draft_Year", #Won't use
    "Grade", #Dummy (Same as above: won't use)
    "Hometown", #Won't use/or international?
    "Team", #won't use
    "High School", #Won't use
    "School", #School from team NCAA data
    "College" #From scouting
]
index = ["Player"]
#Past stats interesting to keep 
to_keep = {
    "avg" : ["average_G", "average_GS", "average_MP", "average_FG", "average_FGA", "average_FG%", "average_3P", "average_3PA", "average_3P%", "average_2P", "average_2PA", "average_2P%", "average_eFG%", "average_FT", "average_FTA", "average_FT%", "average_ORB", "average_DRB", "average_TRB", "average_AST", "average_STL", "average_BLK", "average_TOV", "average_PF", "average_PTS"],		#OK c'est good
    "tot" : ["totals_FG", "totals_FGA",	"totals_FG%", "totals_3P", "totals_3PA", "totals_3P%", "totals_2P",	"totals_2PA", "totals_2P%",	"totals_eFG%", "totals_FT",	"totals_FTA", "totals_FT%",	"totals_ORB", "totals_DRB",	"totals_TRB", "totals_AST",	"totals_STL", "totals_BLK", "totals_TOV", "totals_PF", "totals_PTS"], #OK c'est good
    "per_40" : ["per_40_FG", "per_40_FGA", "per_40_FG%", "per_40_3P", "per_40_3PA",	"per_40_3P%", "per_40_2P", "per_40_2PA", "per_40_2P%", "per_40_eFG%", "per_40_FT", "per_40_FTA", "per_40_FT%", "per_40_ORB", "per_40_DRB", "per_40_TRB", "per_40_AST", "per_40_STL", "per_40_BLK", "per_40_TOV", "per_40_PF", "per_40_PTS"], #OK c'est good
    "scouting_reports" : ["Athleticism", "Size", "Defense", "Strength2", "Quickness", "Leadership", "Jump Shot", "NBA Ready", "Rebounding", "Potential", "Post Skills", "Intangibles"], #OK perfect
    "team" : ["G_team", "W", "L", "W-L%", "SRS", "SOS", #SRS=Simple Rating System
              "W.1", "L.1", "W.2", "L.2", "W.3", "L.3", "Tm.", "Opp.", #1=Conf, 2=Home, 3=Away, Tm./Opp.=Number of points
              "MP_team", "FG_team", "FGA_team", "FG%_team", "3P_team", "3PA_team", "3P%_team", "FT_team", "FTA_team", "FT%_team", "ORB_team", "TRB_team", "AST_team", "STL_team", "BLK_team", "TOV_team", "PF_team"], #How the team behaved, in term of total
    "opp" : ["FG_opp", "FGA_opp", "FG%_opp", "3P_opp", "3PA_opp", "3P%_opp", "FT_opp", "FTA_opp", "FT%_opp", "ORB_opp",	"TRB_opp", "AST_opp", "STL_opp", "BLK_opp",	"TOV_opp", "PF_opp"], #How the opponent team behaved (in term of total)
    "adv" : ["advanced_PER", "advanced_TS%", "advanced_3PAr", "advanced_FTr", "advanced_PProd", "advanced_ORB%", "advanced_DRB%", "advanced_TRB%", "advanced_AST%", "advanced_STL%", "advanced_BLK%", "advanced_TOV%", "advanced_USG%", "advanced_OWS", "advanced_DWS", "advanced_WS", "advanced_WS/40", "advanced_OBPM", "advanced_DBPM", "advanced_BPM"]

}

In [11]:
#Add '_features' to all my columns from the dictionnaries
id = [s + '_features' for s in id] #deleted
team_to_del = [s + '_features' for s in team_to_del] #deleted
adv_to_del = [s + '_features' for s in adv_to_del] #deleted
biometric_data = [s + '_features' for s in biometric_data] 
text_data = [s + '_features' for s in text_data] 
for key in to_keep:
    to_keep[key] = [s + '_features' for s in to_keep[key]]
for count, word in enumerate(useless): #deleted
    try:
        useless[count] = word + '_features'
    except:
        continue

## Missing values

### First look

In [7]:
working_df[text_data + biometric_data].isna().sum()/len(working_df)

Class_features             0.000000
Pos_x_features             0.000000
average_Awards_features    0.823591
Strength_features          0.389353
Weakness_features          0.410230
RSCI Top 100_features      0.419624
Height_features            0.000000
Weight_features            0.000000
Draft_pos_features         0.328810
dtype: float64

In [8]:
for key in to_keep:
    print(working_df[to_keep[key]].isna().sum()/len(working_df))

average_G_features       0.024008
average_GS_features      0.024008
average_MP_features      0.024008
average_FG_features      0.024008
average_FGA_features     0.024008
average_FG%_features     0.024008
average_3P_features      0.024008
average_3PA_features     0.024008
average_3P%_features     0.089770
average_2P_features      0.024008
average_2PA_features     0.024008
average_2P%_features     0.024008
average_eFG%_features    0.024008
average_FT_features      0.024008
average_FTA_features     0.024008
average_FT%_features     0.024008
average_ORB_features     0.024008
average_DRB_features     0.024008
average_TRB_features     0.024008
average_AST_features     0.024008
average_STL_features     0.024008
average_BLK_features     0.024008
average_TOV_features     0.024008
average_PF_features      0.024008
average_PTS_features     0.024008
dtype: float64
totals_FG_features      0.024008
totals_FGA_features     0.024008
totals_FG%_features     0.024008
totals_3P_features      0.024008
tot

Here are the null values:
- ADVANCED : 20 to 23% (before 2010)
- OPPONENT TEAM : 18.3% (before 2010)
- SCOUTING EVAL : 40%
- PER 40 : 2 to 8%
- TOTALS : 2.5%
- AVERAGE : 2.5%
- AWARDS : 82% (**logical**)
- Strength/Weakness : 40% 
- RSCI : 40% (**logical**)
- Draft Pos: 33% (**logical**)

### Dummy variables

In [15]:
#Adding a dummy variable for HS Ranking (RSCI)
working_df['dummy_hs_ranking_features'] = working_df['RSCI Top 100_features'].apply(
    lambda row: 0 if pd.isna(row)  else 1
)
#Adding a dummy variable for Awards
working_df['dummy_coll_awards_features'] = working_df['average_Awards_features'].apply(
    lambda row: 0 if pd.isna(row)  else 1
)
#Adding a dummy variable for Mock draft
working_df['dummy_mock_draft_features'] = working_df['Draft_pos_features'].apply(
    lambda row: 0 if pd.isna(row)  else 1
)
#Adding a dummy variable for Scouting reports
working_df['dummy_scouting_reports_features'] = working_df['Strength_features'].apply(
    lambda row: 0 if pd.isna(row)  else 1
)

In [16]:
from sklearn.preprocessing import OneHotEncoder
#Function for the categorization
def categorize_ranking(row):
    if pd.isna(row) or row == 'NaN':
        return 'Not in the classement'
    else:
        try:
            rank = int(row.split()[0])
        except:
            rank = row
        if 1 <= rank <= 25:
            return '1-25'
        elif 26 <= rank <= 50:
            return '26-50'
        elif 51 <= rank <= 75:
            return '51-75'
        elif 76 <= rank <= 100:
            return '76-100'
        else:
            return 'Not in the classement'

In [17]:
#Create a new column where we apply the function to be able to derive multi classes
## RSCI
working_df['rsci'] = working_df['RSCI Top 100_features'].apply(lambda row: categorize_ranking(row))
#Create a new column where we apply the function to be able to derive multi classes
## Mock Draft
working_df['mock_draft'] = working_df['Draft_pos_features'].apply(lambda row : categorize_ranking(row))

In [18]:
#Create the new columns based on the categories
awards_split = working_df['average_Awards_features'].str.get_dummies(sep=',')
working_df = pd.concat([working_df, awards_split.add_suffix('_features')], axis=1)
#Create the new columns based on the categories
categorical_columns = ['mock_draft', 'rsci', 'Pos_x_features', 'Class_features']
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(working_df[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))
working_df = pd.concat([working_df, one_hot_df.add_suffix('_features')], axis=1)
working_df = working_df.drop(categorical_columns, axis=1)

## Textual Data - Scouting reports

In [19]:
# Fill missing scouting reports with placeholder text
unstructured = working_df[['player_id', 'Strength_features', 'Weakness_features']].copy()
unstructured['text'] = unstructured['Strength_features'] + unstructured['Weakness_features']
unstructured['text'] = unstructured['text'].fillna('No scouting report available') #Fill missing values 

Based on this : https://medium.com/@claude.feldges/text-classification-with-tf-idf-lstm-bert-a-quantitative-comparison-b8409b556cb3

In [20]:
#Function for the cleaning
def utils_preprocess_text(text, flg_stemm=False, flg_lemm=True, lst_stopwords=None):
    # Clean (convert to lowercase and remove punctuations and characters and then strip)
    # The function is not optimized for speed but split into various steps for pedagogical purpose
    text = str(text).lower()
    text = text.strip()
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenize (convert from string to list)
    lst_text = text.split()
    # remove Stopwords
    if lst_stopwords is not None:
        lst_text = [word for word in lst_text if word not in lst_stopwords]

    # Stemming (remove -ing, -ly, ...)
    if flg_stemm == True:
        ps = nltk.stem.porter.PorterStemmer()
        lst_text = [ps.stem(word) for word in lst_text]

    # Lemmatisation (convert the word into root word)
    if flg_lemm == True:
        lem = nltk.stem.wordnet.WordNetLemmatizer()
        lst_text = [lem.lemmatize(word) for word in lst_text]

    # back to string from list
    text = " ".join(lst_text)
    return text

In [21]:
# Text processing
import re
import nltk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# Make sure to download the required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
# Use stopwords list from nltk
lst_stopwords = nltk.corpus.stopwords.words("english")

def utils_preprocess_text(text, flg_stemm=False, flg_lemm=True, lst_stopwords=None):
    # Clean: convert to lowercase, remove punctuations, and strip
    text = str(text).lower()
    text = text.strip()
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenize using nltk's word_tokenize for better handling of punctuation and special characters
    lst_text = word_tokenize(text)
    
    # Remove stopwords using a predefined list if no custom list is provided
    if lst_stopwords is None:
        lst_stopwords = stopwords.words('english')
    lst_text = [word for word in lst_text if word not in lst_stopwords]

    # Stemming (optional)
    if flg_stemm:
        ps = nltk.stem.PorterStemmer()
        lst_text = [ps.stem(word) for word in lst_text]

    # Lemmatization (optional)
    if flg_lemm:
        lem = WordNetLemmatizer()
        lst_text = [lem.lemmatize(word) for word in lst_text]

    # Join the tokens back into a single string
    return " ".join(lst_text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [22]:
# Let's apply this function to the whole corpus
unstructured["clean_2"] = unstructured["text"].apply(lambda x: utils_preprocess_text(x, flg_stemm=False, flg_lemm=True, lst_stopwords=lst_stopwords))
#Add variable to the corpus
corpus = unstructured['clean_2']
max_features = 100
# Initizalize the vectorizer with max nr words and ngrams (1: single words, 2: two words in a row)
vectorizer_tfidf = TfidfVectorizer(stop_words='english', max_features=max_features, ngram_range=(1,2))
# Fit the vectorizer to the training data
X_tfidf = vectorizer_tfidf.fit_transform(corpus)
dense_array = X_tfidf.toarray()
# Convert the dense array into a DataFrame with feature names as columns
tfidf_df = pd.DataFrame(dense_array, columns=vectorizer_tfidf.get_feature_names_out())
tfidf_df = tfidf_df.add_suffix('_features').add_prefix('text_')
#Concatenate everything together
working_df = pd.concat([working_df, tfidf_df], axis=1)

We first try with this, and look at the results. If we don't have satisfaisant results, we will try with more complex models (BERT or Word2Vec).

## Missing values in Past stats

### Opponent

In [23]:
##Configuration
# Get all columns with '_opp' in their name
opp_columns = [col for col in working_df.columns if '_opp' in col]
# Define missing years (to impute) and proxy years (to compute median)
missing_years = (2006, 2009)
proxy_years = (2010, 2025)

In [24]:
# Group by team and impute each '_opp' column
for team in working_df['Team_features'].unique():
    # Filter team data
    team_mask = working_df['Team_features'] == team
    
    # Filter proxy years (2010–2025) for this team
    proxy_data = working_df[team_mask & (working_df['season_features'].between(*proxy_years))]
    
    # Compute median of each '_opp' column for the proxy period
    proxy_medians = proxy_data[opp_columns].median()
    
    # Apply imputation to missing years (2006–2009)
    missing_mask = team_mask & (working_df['season_features'].between(*missing_years))
    working_df.loc[missing_mask, opp_columns] = working_df.loc[missing_mask, opp_columns].fillna(proxy_medians)

In [25]:
#Players that don't have teams after, use data from the stack
RECOVERY_OPP = HOME + W_DB + r"\Features\Team data\4-4-2025_team.xlsx"
opp_recov = pd.read_excel(RECOVERY_OPP)
opp_recov = opp_recov.add_suffix('_features')
# Group by team and impute each '_opp' column
for team in working_df['School_features'].unique():
    # Filter team data
    team_mask = opp_recov['School_features'] == team
    
    # Filter proxy years (2010–2025) for this team
    proxy_data = opp_recov[team_mask & (opp_recov['season_features'].between(*proxy_years))]
    
    # Compute median of each '_opp' column for the proxy period
    proxy_medians = proxy_data[opp_columns].median()
    
    # Apply imputation to missing years (2006–2009)
    missing_mask = team_mask & (working_df['season_features'].between(*missing_years))
    working_df.loc[missing_mask, opp_columns] = working_df.loc[missing_mask, opp_columns].fillna(proxy_medians)

### Average, PER_40 and Totals

In [26]:
stats = ['average', 'per_40', 'totals']
for stat in stats:    
    working_df[f'{stat}_3P%_features'] = working_df[f'{stat}_3P%_features'].fillna(0)
#Remove the row that couldn't get correctly scrapped both for individual and teams stats
row_to_remove = working_df[working_df['average_FTA_features'].isnull(
                        ) | working_df['TRB_opp_features'].isnull()].index
working_df = working_df.drop(index=row_to_remove)
#working_df = working_df.drop('MP_team_features', axis=1)

# First try modelling

We delete the Advanced and the Scouting report part due to missing values, ... And try one by one each types of data (Totals, PER 40 and Average).

In addition, we need to create two variables to store the features and the targets each.

## First test

In [ ]:
varying_stats = to_keep.keys

In [ ]:
model_df = working_df.copy() #Copy the df into in a new variable 
model_df.set_index(index[0] + '_features', inplace=True) #Change the index of the DF to put player names
for i in useless: #Delete useless columns 
    if i != 'player_id_features':
        try:
            model_df = model_df.drop(i, axis=1)
        except:
            for j in i:
                model_df = model_df.drop(j + '_features', axis=1)
    else:
            model_df = model_df.drop(i[0:9], axis=1)
        

In [ ]:
#Vectorizing dataframe based on target chosen 
features = [col for col in model_df.columns if 'features' in col]
metric = 'WS/48'
season = 2
target = f'{metric}-{season}_target'
#Delete the features that we won't use from the list 'features'
#Advanced and Scouting 
to_exclude = to_keep['adv'] + to_keep['scouting_reports']
to_exclude.append('Height_features')
to_exclude.append('RSCI Top 100_features')
to_exclude.append('average_Awards_features')
to_exclude.append('Strength_features')
to_exclude.append('Weakness_features')
features_to_keep = [col for col in features if col not in to_exclude]
model_df[[target] + features_to_keep][model_df[target].notnull()]

In [ ]:
indices = model_df[model_df[target].notnull()].index
X = model_df.loc[indices, features_to_keep]
y = model_df.loc[indices, target]

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor  # ou un autre modèle
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
target = [col for col in working_df.columns if 'target' in col]

In [ ]:
# Initialiser un DataFrame pour stocker les résultats
results_df = pd.DataFrame(columns=['Target', 'MSE', 'Train_R2', 'Test_R2', 'Overfitting'])

# Boucler sur toutes les targets (de target_1 à target_10)
for season in range(1, 11):
    metric = 'WS'
    target_name = f'{metric}-{season}_target'

    # Filtrer les indices où la target est non nulle
    indices = model_df[model_df[target_name].notnull()].index

    # Séparer X (features) et y (target) sur ces indices
    X = model_df.loc[indices, features_to_keep]
    y = model_df.loc[indices, target_name]

    # Diviser en train et test (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

    
     # Création de la pipeline avec standardisation
    pipeline = make_pipeline(
        StandardScaler(),
        RandomForestRegressor(n_estimators=100, random_state=42, max_depth=6)
    )

    pipeline.fit(X_train, y_train)
    # Prédictions sur le train et test
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # Calculer les métriques
    mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Vérifier l'overfitting en comparant le R2 entre train et test
    overfitting = abs(train_r2 - test_r2)

    # Créer un DataFrame temporaire pour les résultats de cette target
    temp_df = pd.DataFrame([{
        'Target': target_name,
        'MSE': mse,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Overfitting': overfitting
    }])

    # Concaténer les résultats dans le DataFrame global
    results_df = pd.concat([results_df, temp_df], ignore_index=True)
    # Afficher un résumé pour chaque target
    print(f"Target: {target_name} - MSE: {mse}, Train R²: {train_r2}, Test R²: {test_r2}, Overfitting: {overfitting}")

# Afficher le DataFrame complet des résultats
print("\nRésumé des résultats pour chaque target:")
print(50*"-")
results_df


In [ ]:
from sklearn.linear_model import LinearRegression


# Boucler sur toutes les targets (de target_1 à target_10)
for season in range(1, 11):
    metric = 'WS'
    target_name = f'{metric}-{season}_target'

    # Filtrer les indices où la target est non nulle
    indices = model_df[model_df[target_name].notnull()].index

    # Séparer X (features) et y (target) sur ces indices
    X = model_df.loc[indices, features_to_keep]
    y = model_df.loc[indices, target_name]

    # Diviser en train et test (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    
    pipeline = make_pipeline(
        StandardScaler(),
        LinearRegression()  # Teste différentes valeurs de alpha
    )
    pipeline.fit(X_train, y_train)
    # Prédictions sur le train et test
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # Calculer les métriques
    mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Vérifier l'overfitting en comparant le R2 entre train et test
    overfitting = abs(train_r2 - test_r2)

    # Créer un DataFrame temporaire pour les résultats de cette target
    temp_df = pd.DataFrame([{
        'Target': target_name,
        'MSE': mse,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Overfitting': overfitting
    }])

    # Concaténer les résultats dans le DataFrame global
    results_df = pd.concat([results_df, temp_df], ignore_index=True)
    # Afficher un résumé pour chaque target
    print(f"Target: {target_name} - MSE: {mse}, Train R²: {train_r2}, Test R²: {test_r2}, Overfitting: {overfitting}")

# Afficher le DataFrame complet des résultats
print("\nRésumé des résultats pour chaque target:")
print(50*"-")
results_df


In [ ]:
for i,j in (working_df[features].isna().sum().sort_values(ascending=False)).items():
    print(i, j)

## Final pipeline

In [27]:
working_df

,Unnamed: 0,player_id,G-1_target,G-2_target,G-3_target,G-4_target,G-5_target,G-6_target,G-7_target,G-8_target,...,text_struggle_features,text_team_features,text_teammate_features,text_throw_features,text_time_features,text_transition_features,text_turnover_features,text_wingspan_features,text_work_features,text_year_features
0,0,aaronbrooks_8,51.0,80.0,82.0,29.5,NaN,26.5,36.0,82.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,aarongordon_15,47.0,78.0,80.0,58.0,78.0,62.0,25.0,75.0,...,0.070105,0.061541,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.138168,0.058714
2,2,aarongray_8,61.0,56.0,16.0,41.0,49.0,42.0,18.5,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,3,aaronharrison_16,21.0,5.0,9.0,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.155207,0.000000,0.000000,0.343804,0.084969,0.088076,0.000000,0.087115,0.074039
4,4,aaronhenry_22,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.096201,0.000000,0.000000,0.085240,0.000000,0.109184,0.093020,0.107993,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,953,zekennaji_21,42.0,41.0,53.0,58.0,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.077461,0.000000,0.000000,0.000000,0.065872,0.076474,0.064995
954,954,zhairesmith_19,6.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.266717,0.000000,0.202037,0.062415,0.000000,0.055120,0.063992,0.000000
955,955,ziairewilliams_22,62.0,37.0,51.0,NaN,NaN,NaN,NaN,NaN,...,0.063868,0.056066,0.000000,0.000000,0.049677,0.000000,0.063632,0.054212,0.000000,0.053490
956,956,zionwilliamson_20,24.0,61.0,NaN,29.0,70.0,NaN,NaN,NaN,...,0.100625,0.132499,0.051662,0.050219,0.195670,0.096716,0.000000,0.042706,0.247899,0.000000


In [ ]:
from preProcessingAuto import preprocess_working_df
from modelAutomation import get_features_and_target, clean_features, evaluate_models, prepare_features
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor  # ou un autre modèle
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from config import text_data, biometric_data, adv_to_del, team_to_del, id, useless, index, to_keep

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
RECOVERY_OPP = HOME + W_DB + r"\Features\Team data\4-4-2025_team.xlsx"
#Now run the full preprocessing:
processed_df = preprocess_working_df(working_df, recovery_file=RECOVERY_OPP)

2025-04-15 16:44:43,306 - INFO - Starting full preprocessing pipeline...
2025-04-15 16:44:43,306 - INFO - Creating dummy variables...
2025-04-15 16:44:43,313 - INFO - After dummy variable creation: shape = (958, 364)
2025-04-15 16:44:43,314 - INFO - Categorizing and one-hot encoding categorical features...
2025-04-15 16:44:43,327 - INFO - After categorization and one-hot encoding: shape = (958, 379)
2025-04-15 16:44:43,328 - INFO - Processing unstructured text data...
2025-04-15 16:44:46,537 - INFO - After TF-IDF vectorization: shape = (958, 100)
2025-04-15 16:44:46,556 - INFO - After after TF-IDF merge: shape = (958, 479)
2025-04-15 16:44:46,556 - INFO - Imputing opponent columns based on team data...
2025-04-15 16:44:47,955 - INFO - After imputation using team data: shape = (958, 479)
2025-04-15 16:44:47,955 - INFO - Imputing opponent columns using recovery data...
2025-04-15 16:44:52,736 - INFO - After imputation using recovery data: shape = (958, 479)
2025-04-15 16:44:52,736 - INFO

,Unnamed: 0,player_id,G-1_target,G-2_target,G-3_target,G-4_target,G-5_target,G-6_target,G-7_target,G-8_target,...,text_team_features,text_teammate_features,text_throw_features,text_time_features,text_transition_features,text_turnover_features,text_wing_features,text_wingspan_features,text_work_features,text_year_features
0,0,aaronbrooks_8,51.0,80.0,82.0,29.5,NaN,26.5,36.0,82.0,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000
1,1,aarongordon_15,47.0,78.0,80.0,58.0,78.0,62.0,25.0,75.0,...,0.061500,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.00000,0.138721,0.058775
2,2,aarongray_8,61.0,56.0,16.0,41.0,49.0,42.0,18.5,NaN,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000
3,3,aaronharrison_16,21.0,5.0,9.0,NaN,NaN,NaN,NaN,NaN,...,0.154524,0.0,0.0,0.343419,0.08499,0.087936,0.099486,0.00000,0.087137,0.073839
4,4,aaronhenry_22,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.095866,0.0,0.0,0.085223,0.00000,0.109111,0.000000,0.09029,0.108119,0.000000


In [6]:
processed_df

,Unnamed: 0,player_id,G-1_target,G-2_target,G-3_target,G-4_target,G-5_target,G-6_target,G-7_target,G-8_target,...,text_team_features,text_teammate_features,text_throw_features,text_time_features,text_transition_features,text_turnover_features,text_wing_features,text_wingspan_features,text_work_features,text_year_features
0,0,aaronbrooks_8,51.0,80.0,82.0,29.5,NaN,26.5,36.0,82.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,aarongordon_15,47.0,78.0,80.0,58.0,78.0,62.0,25.0,75.0,...,0.061500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.138721,0.058775
2,2,aarongray_8,61.0,56.0,16.0,41.0,49.0,42.0,18.5,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,3,aaronharrison_16,21.0,5.0,9.0,NaN,NaN,NaN,NaN,NaN,...,0.154524,0.000000,0.000000,0.343419,0.084990,0.087936,0.099486,0.000000,0.087137,0.073839
4,4,aaronhenry_22,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.095866,0.000000,0.000000,0.085223,0.000000,0.109111,0.000000,0.090290,0.108119,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,953,zekennaji_21,42.0,41.0,53.0,58.0,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.077803,0.000000,0.000000,0.000000,0.000000,0.064145,0.076812,0.065089
954,954,zhairesmith_19,6.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.265770,0.000000,0.201432,0.062313,0.000000,0.072942,0.053352,0.063887,0.000000
955,955,ziairewilliams_22,62.0,37.0,51.0,NaN,NaN,NaN,NaN,NaN,...,0.053701,0.000000,0.000000,0.047739,0.000000,0.061121,0.276594,0.050577,0.000000,0.051322
956,956,zionwilliamson_20,24.0,61.0,NaN,29.0,70.0,NaN,NaN,NaN,...,0.132017,0.051615,0.050270,0.195599,0.096814,0.000000,0.000000,0.041446,0.248150,0.000000


In [15]:
working_df = processed_df.copy()
# Get initial features and targets
raw_features, target = get_features_and_target(working_df)


# Step 2: Prepare feature list (optionally re-adding 'adv' and 'scouting_reports')
include_subsets = ['per_40', 'team', 'opp']
features, included_types = prepare_features(
    base_features=raw_features,
    df=working_df,
    useless_list=useless,
    feature_dict=to_keep,
    include_keys=include_subsets
)
# Define the set of models to test

models_dict = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),  # you can tune alpha
    "Lasso": Lasso(alpha=0.1),  # same here
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "KNeighbors": KNeighborsRegressor(n_neighbors=5)
}
# Define scalers to try
scalers_dict = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "Normalizer": Normalizer()
}
test_size = [0.1, 0.15, 0.2]

dic_metr = ['WS', 'WS/48']

# Evaluate models
results = evaluate_models(df=working_df,
                          features=features,
                          metric=dic_metr[1],
                          season_range=range(1, 3),
                          index_col='Player_features',
                          drop_cols=['Height_features'],  # if you want to drop specific features
                          test_sizes=test_size,
                          models_dict=models_dict, 
                          scalers_dict=scalers_dict,
                          feature_types=include_subsets
                          )

# Show the aggregated results
print("\nSummary of evaluations:")
results.sort_values(ascending=False, by='Test_R2')

2025-04-15 16:47:21,671 - INFO - Evaluating Model: RandomForest, Scaler: StandardScaler, Test Size: 0.1
c:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Thesis - Code\modelAutomation.py:161: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([temp_dict])], ignore_index=True)
2025-04-15 16:47:22,881 - INFO - Target: WS/48-1_target | MSE: 0.011 | Train R²: 0.219 | Test R²: 0.124 | Overfitting: 0.095
2025-04-15 16:47:23,839 - INFO - Target: WS/48-2_target | MSE: 0.006 | Train R²: 0.323 | Test R²: 0.001 | Overfitting: 0.322
2025-04-15 16:47:23,840 - INFO - Evaluating Model: LinearRegression, Scaler: StandardScaler, Test Size: 0.1
2025-04-15 16:47:23,953 - INFO - Target: WS/48-1_target | MSE: 0.014 | Tr


Summary of evaluations:


,Target,Model,Scaler,Test_Size,Feature_Types,MSE,Train_R2,Test_R2,Overfitting
24,WS/48-1_target,RandomForest,Normalizer,0.10,"per_40, team, opp",1.116624e-02,0.226196,1.408191e-01,8.537663e-02
0,WS/48-1_target,RandomForest,StandardScaler,0.10,"per_40, team, opp",1.138422e-02,0.219217,1.240465e-01,9.517059e-02
12,WS/48-1_target,RandomForest,MinMaxScaler,0.10,"per_40, team, opp",1.139259e-02,0.219233,1.234027e-01,9.583039e-02
60,WS/48-1_target,RandomForest,Normalizer,0.15,"per_40, team, opp",9.990742e-03,0.223392,1.228011e-01,1.005909e-01
32,WS/48-1_target,GradientBoosting,Normalizer,0.10,"per_40, team, opp",1.152432e-02,0.713168,1.132662e-01,5.999021e-01
...,...,...,...,...,...,...,...,...,...
74,WS/48-1_target,LinearRegression,StandardScaler,0.20,"per_40, team, opp",1.463444e+17,0.301164,-1.296430e+19,1.296430e+19
87,WS/48-2_target,LinearRegression,MinMaxScaler,0.20,"per_40, team, opp",9.818392e+17,0.371345,-5.457497e+19,5.457497e+19
51,WS/48-2_target,LinearRegression,MinMaxScaler,0.15,"per_40, team, opp",4.206449e+19,0.283747,-5.832366e+21,5.832366e+21
50,WS/48-1_target,LinearRegression,MinMaxScaler,0.15,"per_40, team, opp",1.187779e+20,0.241093,-1.042883e+22,1.042883e+22
